# DINO Mask R-CNN Feature Diagnostic

This notebook inspects the current Toy DINO + TorchVision Mask R-CNN path. It does not train. It loads one instance-segmentation batch, builds the Toy DINO Mask R-CNN model, and summarizes the feature pyramid emitted by the DINO backbone.

Use this on HPC to decide whether the Toy DINO feature contract can be adapted cleanly to a TerraTorch Mask R-CNN path.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import torch

# Set this to the repo root on HPC.
REPO_ROOT = Path(r"/explore/nobackup/people/ajkerr1/Lunar_FM/full_model_lfm/lfm")
DATA_ROOT = Path(r"/explore/nobackup/projects/lfm/model_inputs/300_300_inputs/full_model_inst_seg_v2")

# Optional local DINO checkpoint. If None, the current loader default is used.
DINO_CHECKPOINT = None

OUTPUT_JSON = REPO_ROOT / "scripts" / "outputs" / "dino_mask_rcnn_feature_diagnostic.json"

TARGET_SIZE = 256
BAND_FILTER = [0, 1, 2, 3, 4, 5, 6]
IMAGE_GLOB = "*.tif"
LABEL_GLOB = "*_label.npz"
IMAGE_SUFFIX = "_input_wac_chip"
LABEL_SUFFIX = "_label"
BATCH_SIZE = 2
MAX_SAMPLES = 5
ANCHOR_SIZES = [[8], [16], [32], [64]]
ANCHOR_ASPECT_RATIOS = [0.5, 1.0, 2.0]
FREEZE_BACKBONE = False

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("repo root:", REPO_ROOT)
print("data root:", DATA_ROOT)
print("cuda available:", torch.cuda.is_available())

In [ ]:
from lfm.toy_model.inst_seg.iseg_model import load_dinov3_encoder
from lfm.toy_model.inst_seg.dino_mask_rcnn_model import create_dino_mask_rcnn_model
from lfm.toy_model.inst_seg.lightning_wrappers import ToyDinoMaskRCNNSplitDataModule

dm = ToyDinoMaskRCNNSplitDataModule(
    data_root=DATA_ROOT,
    batch_size=BATCH_SIZE,
    num_workers=0,
    target_size=TARGET_SIZE,
    image_glob=IMAGE_GLOB,
    label_glob=LABEL_GLOB,
    image_suffix=IMAGE_SUFFIX,
    label_suffix=LABEL_SUFFIX,
    band_filter=BAND_FILTER,
    normalize_inputs=False,
    scale_inputs=True,
    mask_shift=(0, 0),
    max_train_samples=MAX_SAMPLES,
    max_val_samples=MAX_SAMPLES,
    max_test_samples=MAX_SAMPLES,
)
dm.setup("fit")
batch = next(iter(dm.train_dataloader()))

print("weight assignments:", dm.weight_assignments)
print("batch keys:", sorted(batch.keys()))
print("image batch shape:", tuple(batch["image"].shape))
print("first target boxes:", tuple(batch["boxes"][0].shape))
print("first target masks:", tuple(batch["masks"][0].shape))

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
encoder_kwargs = {"device": device}
if DINO_CHECKPOINT is not None:
    encoder_kwargs["weights_local_checkpoint"] = str(DINO_CHECKPOINT)

encoder = load_dinov3_encoder(**encoder_kwargs)
model = create_dino_mask_rcnn_model(
    encoder=encoder,
    num_bands=len(dm.weight_assignments),
    target_size=TARGET_SIZE,
    weight_assignments=dm.weight_assignments,
    freeze_backbone=FREEZE_BACKBONE,
    anchor_sizes=ANCHOR_SIZES,
    anchor_aspect_ratios=ANCHOR_ASPECT_RATIOS,
).to(device)
model.eval()

print("model type:", type(model))
print("backbone type:", type(model.backbone))
print("backbone out_channels:", getattr(model.backbone, "out_channels", None))

In [ ]:
def summarize_tensor(tensor: torch.Tensor) -> dict:
    tensor = tensor.detach().cpu()
    return {
        "shape": list(tensor.shape),
        "dtype": str(tensor.dtype),
        "min": float(tensor.min().item()) if tensor.numel() else None,
        "max": float(tensor.max().item()) if tensor.numel() else None,
        "mean": float(tensor.float().mean().item()) if tensor.numel() else None,
        "std": float(tensor.float().std(unbiased=False).item()) if tensor.numel() else None,
    }

images = batch["image"].to(device)
with torch.no_grad():
    features = model.backbone(images)

input_h, input_w = images.shape[-2:]
feature_summary = {}
for name, feature in features.items():
    _, channels, height, width = feature.shape
    feature_summary[name] = {
        **summarize_tensor(feature),
        "channels": int(channels),
        "approx_stride_h": float(input_h / height),
        "approx_stride_w": float(input_w / width),
    }

diagnostic = {
    "data_root": str(DATA_ROOT),
    "target_size": TARGET_SIZE,
    "band_filter": BAND_FILTER,
    "weight_assignments": dm.weight_assignments,
    "batch_image_shape": list(images.shape),
    "backbone_type": f"{type(model.backbone).__module__}.{type(model.backbone).__name__}",
    "backbone_out_channels": getattr(model.backbone, "out_channels", None),
    "feature_names": list(features.keys()),
    "features": feature_summary,
    "anchor_sizes": ANCHOR_SIZES,
    "anchor_aspect_ratios": ANCHOR_ASPECT_RATIOS,
}

print(json.dumps(diagnostic, indent=2))
OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_JSON.write_text(json.dumps(diagnostic, indent=2), encoding="utf-8")
print("wrote", OUTPUT_JSON)

## How To Interpret

For TerraTorch Mask R-CNN feasibility, the important fields are `feature_names`, each feature tensor shape, `backbone_out_channels`, and the approximate strides. A clean adapter is more likely if the Toy DINO backbone can reliably emit a feature dictionary with stable names, equal channel counts, and strides compatible with the detector neck/head configuration.